# LLM Zoomcamp 2026 – Homework 5: Monitoring

This notebook implements the Homework 5 tasks using OpenTelemetry and SQLite.

**Required files in the same folder:**
- `starter.py`
- `rag_helper.py`
- `.env` containing `OPENAI_API_KEY=...`


## 0. Install dependencies in the active notebook kernel

In [1]:
%pip install -q opentelemetry-api opentelemetry-sdk pandas python-dotenv openai minsearch gitsource
print("Dependencies installed. Restart the kernel once if imports still fail.")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dependencies installed. Restart the kernel once if imports still fail.


## 1. Check Python environment and files

In [2]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("starter.py exists:", Path("starter.py").exists())
print("rag_helper.py exists:", Path("rag_helper.py").exists())
print(".env exists:", Path(".env").exists())

Python: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python
Working directory: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/05-monitoring/llm-zoomcamp-hw5
starter.py exists: True
rag_helper.py exists: True
.env exists: True


In [3]:
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in .env"
print("OPENAI_API_KEY loaded successfully.")

OPENAI_API_KEY loaded successfully.


## 2. OpenTelemetry console exporter

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# A provider can only be registered globally once per kernel.
# Restart the kernel before re-running this entire notebook from the top.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

print("OpenTelemetry console exporter configured.")

OpenTelemetry console exporter configured.


## 3. Load the starter RAG

In [5]:
from pathlib import Path

print("Aktueller Ordner:", Path.cwd())
print("Dateien:", [p.name for p in Path.cwd().iterdir()])

Aktueller Ordner: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/05-monitoring/llm-zoomcamp-hw5
Dateien: ['uv.lock', 'pyproject.toml', 'homework_hw5.ipynb', '__pycache__', 'traces.db', 'README.md', 'rag_helper.py', '.env', '.venv', '.python-version', '.ipynb_checkpoints', 'homework5.ipynb', 'main.py', 'starter.py']


In [6]:
import inspect
import starter

rag = starter.rag
RAGBase = type(rag)

print("RAG class:", RAGBase)
print("RAG attributes:", sorted(vars(rag).keys()))
print("rag() signature:", inspect.signature(rag.rag))
print("search() signature:", inspect.signature(rag.search))
print("llm() signature:", inspect.signature(rag.llm))

RAG class: <class 'rag_helper.RAGBase'>
RAG attributes: ['index', 'instructions', 'llm_client', 'model', 'prompt_template']
rag() signature: (query)
search() signature: (query, num_results=5)
llm() signature: (prompt)


## Q1. First trace

Create one span for each of these methods:
- `rag`
- `search`
- `llm`

**Question:** How many spans does the trace produce?


In [7]:
class RAGTraced(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm"):
            return super().llm(*args, **kwargs)

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


# Reuse the already initialized starter object without guessing constructor arguments.
traced_rag = object.__new__(RAGTraced)
traced_rag.__dict__.update(rag.__dict__)

print("RAGTraced created successfully.")

RAGTraced created successfully.


In [8]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)
print(answer)

print("\nQ1: Count the span entries named search, llm, and rag in the output above.")

{
    "name": "search",
    "context": {
        "trace_id": "0x2de715064f727774abf5618354629c53",
        "span_id": "0xbe7d21030964b61f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xafb68c12fd3f764b",
    "start_time": "2026-07-26T15:19:50.028216Z",
    "end_time": "2026-07-26T15:19:50.031750Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ca3ac4b3-a12a-4a3f-b7ed-9d9b766d0b43",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x2de715064f727774abf5618354629c53",
        "span_id": "0x1fe2c1af962fdd50",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xafb68c12fd3f764b",
    "start_time": "2026-07-26T15:19:50.033845Z",
    "end_time": "2026-07-26T15:19:52.649807Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ca3ac4b3-a12a-4a3f-b7ed-9d9b766d0b43",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0x2de715064f727774abf5618354629c53",
        "span_id": "0xafb68c12fd3f764b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.IN

## Q2. Capture token usage and cost

**Question: How many input tokens do we see for the LLM call?**

In [9]:
# Adjust these prices only if your selected model uses different rates.
# Values are USD per 1,000,000 tokens.
INPUT_PRICE_PER_MILLION = 0.0
OUTPUT_PRICE_PER_MILLION = 0.0


class RAGTracedWithMetrics(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(*args, **kwargs)

            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)

                if input_tokens is not None:
                    span.set_attribute("input_tokens", input_tokens)
                if output_tokens is not None:
                    span.set_attribute("output_tokens", output_tokens)

                if input_tokens is not None and output_tokens is not None:
                    cost = (
                        input_tokens * INPUT_PRICE_PER_MILLION
                        + output_tokens * OUTPUT_PRICE_PER_MILLION
                    ) / 1_000_000
                    span.set_attribute("cost", cost)

            return response

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


traced_rag_metrics = object.__new__(RAGTracedWithMetrics)
traced_rag_metrics.__dict__.update(rag.__dict__)

print("Metric-enabled RAG created.")

Metric-enabled RAG created.


In [10]:
answer = traced_rag_metrics.rag(query)
print(answer)

print("\nQ2: Read input_tokens from the llm span output and select the closest option.")

{
    "name": "search",
    "context": {
        "trace_id": "0x303c921aa753adec4cac802b662bd875",
        "span_id": "0x07a31b95c9c5ce61",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8cf303ba32a962a8",
    "start_time": "2026-07-26T15:19:52.677868Z",
    "end_time": "2026-07-26T15:19:52.681321Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ca3ac4b3-a12a-4a3f-b7ed-9d9b766d0b43",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x303c921aa753adec4cac802b662bd875",
        "span_id": "0x208397c59aac2363",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q3. Span timing

In the console output, compare `start_time` and `end_time` for the `llm` span. The later SQLite analysis will calculate durations automatically.

**Question: For a typical query, roughly how long does the LLM call take?**

In [18]:
from datetime import datetime

llm_start = "2026-07-26T15:15:40.206256Z"
llm_end = "2026-07-26T15:15:42.242619Z"

start_time = datetime.fromisoformat(llm_start.replace("Z", "+00:00"))
end_time = datetime.fromisoformat(llm_end.replace("Z", "+00:00"))

duration_ms = (end_time - start_time).total_seconds() * 1000

print(f"LLM duration: {duration_ms:.2f} ms")

if duration_ms < 100:
    answer = "Under 100ms"
elif duration_ms < 500:
    answer = "100-500ms"
elif duration_ms <= 2000:
    answer = "500-2000ms"
else:
    answer = "Over 2000ms"

print(f"Q3 answer: {answer}")

LLM duration: 2036.36 ms
Q3 answer: Over 2000ms


## Q4. Save spans to SQLite

**Question: Which span names appear in the spans table?**

In [ ]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)

        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)

        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})

            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )

        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        self.conn.commit()
        return True


print("SQLiteSpanExporter defined.")

SQLiteSpanExporter defined.


### Important

Restart the kernel now, then run the next setup cell instead of the earlier console-provider cell. OpenTelemetry allows only one global provider per kernel.

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

sqlite_provider = TracerProvider()

sqlite_provider.add_span_processor(
    SimpleSpanProcessor(
        SQLiteSpanExporter("traces.db")
    )
)

trace.set_tracer_provider(sqlite_provider)

tracer = trace.get_tracer("llm-zoomcamp")

print("SQLite exporter configured.")

NameError: name 'SQLiteSpanExporter' is not defined

In [20]:
# Recreate the metric-enabled object after the SQLite tracer is active.
traced_rag_metrics = object.__new__(RAGTracedWithMetrics)
traced_rag_metrics.__dict__.update(rag.__dict__)

answer = traced_rag_metrics.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x4a548867f7df30cb1b29e3cba4978e83",
        "span_id": "0xe7bec6c45e18dc76",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x675477128bb95195",
    "start_time": "2026-07-26T15:25:57.497538Z",
    "end_time": "2026-07-26T15:25:57.500801Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ca3ac4b3-a12a-4a3f-b7ed-9d9b766d0b43",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x4a548867f7df30cb1b29e3cba4978e83",
        "span_id": "0x9f5e7660714a263f",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [21]:
import pandas as pd
import sqlite3

with sqlite3.connect("traces.db") as conn:
    spans_df = pd.read_sql_query("SELECT * FROM spans", conn)

display(spans_df)
print("Q4 span names:", sorted(spans_df["name"].dropna().unique().tolist()))

,name,start_time,end_time,input_tokens,output_tokens,cost


Q4 span names: []


## Q5. Which child span takes the most total time?

In [15]:
spans_df["duration_ms"] = (
    spans_df["end_time"] - spans_df["start_time"]
) / 1_000_000

duration_summary = (
    spans_df.loc[spans_df["name"] != "rag"]
    .groupby("name", as_index=False)["duration_ms"]
    .sum()
    .sort_values("duration_ms", ascending=False)
)

display(duration_summary)

if not duration_summary.empty:
    print("Q5 answer:", duration_summary.iloc[0]["name"])

,name,duration_ms


## Q6. Token stability across four runs

In [16]:
# The database already contains at least one call from Q4.
# Run the same query three additional times.
for run_number in range(1, 4):
    print(f"Run {run_number}/3")
    traced_rag_metrics.rag(query)

print("Three additional calls completed.")

Run 1/3
{
    "name": "search",
    "context": {
        "trace_id": "0xac4e2830db7bc013ae85caf750f231f9",
        "span_id": "0x5862be6ede5f8376",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0d9f8b930bd4f9f5",
    "start_time": "2026-07-26T15:19:55.208467Z",
    "end_time": "2026-07-26T15:19:55.213647Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "ca3ac4b3-a12a-4a3f-b7ed-9d9b766d0b43",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xac4e2830db7bc013ae85caf750f231f9",
        "span_id": "0x8548959934c23579",
        "trace_state": "[]"
    },
    "kind": "

In [17]:
with sqlite3.connect("traces.db") as conn:
    llm_tokens = pd.read_sql_query(
        """
        SELECT rowid, input_tokens
        FROM spans
        WHERE name = 'llm' AND input_tokens IS NOT NULL
        ORDER BY rowid DESC
        LIMIT 4
        """,
        conn,
    ).sort_values("rowid")

display(llm_tokens)

tokens = llm_tokens["input_tokens"].astype(float)

if len(tokens) == 4:
    minimum = tokens.min()
    maximum = tokens.max()
    variation = 0.0 if minimum == 0 else (maximum - minimum) / minimum

    print(f"Minimum: {minimum:.0f}")
    print(f"Maximum: {maximum:.0f}")
    print(f"Variation: {variation:.2%}")

    if variation == 0:
        answer_q6 = "They're identical"
    elif variation <= 0.10:
        answer_q6 = "Within 10% of each other"
    elif variation <= 0.50:
        answer_q6 = "Within 50% of each other"
    else:
        answer_q6 = "They vary more than 50%"

    print("Q6 answer:", answer_q6)
else:
    print("Expected 4 llm rows, but found:", len(tokens))

,rowid,input_tokens


Expected 4 llm rows, but found: 0


## Final answers

Fill these values from your own outputs:

1. Number of spans: `...`
2. Input tokens: `...`
3. Typical LLM duration: `...`
4. Span names in SQLite: `...`
5. Slowest child span: `...`
6. Input-token variation: `...`
